In [ ]:
import numpy as np
from datetime import datetime

In [ ]:
file_path = "hostel_bois.txt"

with open(file_path, "r", encoding="utf-8") as file:
    lines = file.readlines()

print("Total Lines in File :", len(lines))

Total Lines in File : 3178


In [ ]:
messages = []

system_messages = 0
media_messages = 0
deleted_messages = 0

In [ ]:
for line in lines:

    line = line.strip()

    if line == "":
        continue


    if len(line) < 8:
        continue

    if line[2] != "/" or line[5] != "/":
        continue

    try:

        timestamp, remaining = line.split(" - ", 1)

        sender, message = remaining.split(": ", 1)

    except ValueError:

        system_messages += 1
        continue



    if message == "<Media omitted>":
        media_messages += 1


    if message == "This message was deleted":
        deleted_messages += 1


    messages.append({
        "timestamp": timestamp,
        "sender": sender,
        "message": message
    })

In [ ]:
print("First 5 Messages\n")

for msg in messages[:5]:
    print(msg)

First 5 Messages

{'timestamp': '01/04/24, 01:17', 'sender': 'Rahul', 'message': 'scene fix'}
{'timestamp': '01/04/24, 01:17', 'sender': 'Rahul', 'message': 'haan'}
{'timestamp': '01/04/24, 01:18', 'sender': 'Rahul', 'message': 'kya scene'}
{'timestamp': '01/04/24, 02:13', 'sender': 'Rahul', 'message': 'abhi free hai?'}
{'timestamp': '01/04/24, 02:13', 'sender': 'Rahul', 'message': 'abey'}


In [ ]:
participants = set()

for msg in messages:
    participants.add(msg["sender"])

print("="*60)
print("CHAT PARSER")
print("="*60)

print("Parsed Messages      :", len(messages))
print("Participants         :", len(participants))
print("System Messages      :", system_messages)
print("Media Messages       :", media_messages)
print("Deleted Messages     :", deleted_messages)

CHAT PARSER
Parsed Messages      : 3174
Participants         : 6
System Messages      : 4
Media Messages       : 32
Deleted Messages     : 15


In [ ]:
person_count = {}

for msg in messages:

    sender = msg["sender"]

    if sender not in person_count:
        person_count[sender] = 0

    person_count[sender] += 1

sorted_people = sorted(person_count.items(),
                       key=lambda x: x[1],
                       reverse=True)

print("="*60)
print("GROUP OVERVIEW")
print("="*60)

print("Group Name       : Hostel Bois 4ever")
print("Total Messages   :", len(messages))
print("Participants     :", len(participants))

print("\nMessages Per Person\n")

for name, count in sorted_people:

    percentage = (count/len(messages))*100

    print(f"{name:10} : {count:4} ({percentage:.1f}%)")

GROUP OVERVIEW
Group Name       : Hostel Bois 4ever
Total Messages   : 3174
Participants     : 6

Messages Per Person

Rahul      :  953 (30.0%)
Priya      :  718 (22.6%)
Neha       :  635 (20.0%)
Aman       :  490 (15.4%)
Karan      :  354 (11.2%)
Vikas      :   24 (0.8%)


In [ ]:
day_count = {}

for msg in messages:

    date = msg["timestamp"].split(",")[0]

    if date not in day_count:
        day_count[date] = 0

    day_count[date] += 1

In [ ]:
busiest_day = max(day_count, key=day_count.get)

print("="*60)
print("BUSIEST DAY")
print("="*60)

print("Date :", busiest_day)
print("Messages :", day_count[busiest_day])

BUSIEST DAY
Date : 04/05/24
Messages : 76


In [ ]:
hour_count = {}

for i in range(24):
    hour_count[i] = 0

for msg in messages:

    time = msg["timestamp"].split(", ")[1]

    hour = int(time.split(":")[0])

    hour_count[hour] += 1

In [ ]:
busiest_hour = max(hour_count, key=hour_count.get)

print("="*60)
print("BUSIEST HOUR")
print("="*60)

print(f"{busiest_hour:02}:00 - {busiest_hour+1:02}:00")
print("Messages :", hour_count[busiest_hour])

BUSIEST HOUR
18:00 - 19:00
Messages : 248


In [ ]:
people = sorted(list(participants))

person_index = {}

for i in range(len(people)):
    person_index[people[i]] = i

print(person_index)

{'Aman': 0, 'Karan': 1, 'Neha': 2, 'Priya': 3, 'Rahul': 4, 'Vikas': 5}


In [ ]:
heatmap = np.zeros((len(people), 24), dtype=int)

for msg in messages:

    sender = msg["sender"]

    hour = int(msg["timestamp"].split(", ")[1].split(":")[0])

    row = person_index[sender]

    heatmap[row][hour] += 1

print(heatmap)

[[ 54  67  66  60  88   0   0   0   0   0   0   0   0   0  14  11  19   7
   16   8  13  11   0  56]
 [  0   0   0   0   0   0   0   4  12  16  20  16  37  25  32  27  27  27
   25  32  23  14   9   8]
 [  0   0   0   0   0  19   3  13  36  52  52  22  39  36  27  10  37  47
   62  50  45  27  28  30]
 [  0   0   0   0   0   0  13  20  47  65  62  61  57  48  44  29  32  40
   38  60  43  32  18   9]
 [  3  15  17  17  22  10  17  17  24  17  25  15  58  48  45  53  73  49
  105  76  41  92  60  54]
 [  0   0   0   0   0   0   0   1   3   1   1   0   2   2   0   1   1   3
    2   2   1   1   1   2]]


In [ ]:
print("="*80)
print("ACTIVITY HEATMAP")
print("="*80)

print("     ", end="")

for h in range(24):
    print(f"{h:02}", end=" ")

print()

for i in range(len(people)):

    print(f"{people[i]:8}", end=" ")

    maximum = max(heatmap[i])

    for value in heatmap[i]:

        if maximum == 0:
            symbol = "."

        else:

            ratio = value / maximum

            if ratio == 0:
                symbol = "."

            elif ratio <= 0.25:
                symbol = "░"

            elif ratio <= 0.50:
                symbol = "▒"

            elif ratio <= 0.75:
                symbol = "▓"

            else:
                symbol = "█"

        print(symbol, end="  ")

    print()

ACTIVITY HEATMAP
     00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23 
Aman     ▓  █  ▓  ▓  █  .  .  .  .  .  .  .  .  .  ░  ░  ░  ░  ░  ░  ░  ░  .  ▓  
Karan    .  .  .  .  .  .  .  ░  ▒  ▒  ▓  ▒  █  ▓  █  ▓  ▓  ▓  ▓  █  ▓  ▒  ░  ░  
Neha     .  .  .  .  .  ▒  ░  ░  ▓  █  █  ▒  ▓  ▓  ▒  ░  ▓  █  █  █  ▓  ▒  ▒  ▒  
Priya    .  .  .  .  .  .  ░  ▒  ▓  █  █  █  █  ▓  ▓  ▒  ▒  ▓  ▓  █  ▓  ▒  ▒  ░  
Rahul    ░  ░  ░  ░  ░  ░  ░  ░  ░  ░  ░  ░  ▓  ▒  ▒  ▓  ▓  ▒  █  ▓  ▒  █  ▓  ▓  
Vikas    .  .  .  .  .  .  .  ▒  █  ▒  ▒  .  ▓  ▓  .  ▒  ▒  █  ▓  ▓  ▒  ▒  ▒  ▓  


In [ ]:
print("="*60)
print("TOTAL MESSAGES EACH HOUR")
print("="*60)

for hour in range(24):

    total = np.sum(heatmap[:, hour])

    print(f"{hour:02}:00 - {hour+1:02}:00  : {total}")

TOTAL MESSAGES EACH HOUR
00:00 - 01:00  : 57
01:00 - 02:00  : 82
02:00 - 03:00  : 83
03:00 - 04:00  : 77
04:00 - 05:00  : 110
05:00 - 06:00  : 29
06:00 - 07:00  : 33
07:00 - 08:00  : 55
08:00 - 09:00  : 122
09:00 - 10:00  : 151
10:00 - 11:00  : 160
11:00 - 12:00  : 114
12:00 - 13:00  : 193
13:00 - 14:00  : 159
14:00 - 15:00  : 162
15:00 - 16:00  : 131
16:00 - 17:00  : 189
17:00 - 18:00  : 173
18:00 - 19:00  : 248
19:00 - 20:00  : 228
20:00 - 21:00  : 166
21:00 - 22:00  : 177
22:00 - 23:00  : 116
23:00 - 24:00  : 159


In [ ]:
stop_words = [
    "i","me","my","we","our","you","your",
    "is","am","are","was","were","the","a",
    "an","and","or","to","of","in","on","for",
    "it","its","this","that","be","have","has",
    "had","do","does","did","with","at","as",
    "from","by","if","but","so","not","he",
    "she","they","them","his","her","their"
]

In [ ]:
word_count = {}

punctuation = ".,!?;:'\"()[]{}<>-/\\@#$%^&*_+=|`~"

for msg in messages:

    text = msg["message"]


    if text == "<Media omitted>":
        continue


    if text == "This message was deleted":
        continue

    words = text.split()

    for word in words:

        word = word.lower()

        word = word.strip(punctuation)

        if word == "":
            continue

        if word in stop_words:
            continue

        if word not in word_count:
            word_count[word] = 0

        word_count[word] += 1

In [ ]:
top_words = sorted(
    word_count.items(),
    key=lambda x: x[1],
    reverse=True
)

print(top_words[:10])

[('how', 321), ('guys', 318), ('about', 274), ('hai', 268), ('today', 257), ('just', 208), ('which', 202), ('everyone', 187), ('telling', 179), ('up', 172)]


In [ ]:
print("="*60)
print("TOP 10 WORDS")
print("="*60)

maximum = top_words[0][1]

for word, count in top_words[:10]:

    bar = "█" * int((count / maximum) * 25)

    print(f"{word:15} {bar} {count}")

TOP 10 WORDS
how             █████████████████████████ 321
guys            ████████████████████████ 318
about           █████████████████████ 274
hai             ████████████████████ 268
today           ████████████████████ 257
just            ████████████████ 208
which           ███████████████ 202
everyone        ██████████████ 187
telling         █████████████ 179
up              █████████████ 172


In [ ]:
person_length = {}
person_messages = {}

for msg in messages:

    sender = msg["sender"]
    text = msg["message"]

    if text == "<Media omitted>":
        continue

    if text == "This message was deleted":
        continue

    length = len(text)

    if sender not in person_length:
        person_length[sender] = 0
        person_messages[sender] = 0

    person_length[sender] += length
    person_messages[sender] += 1

In [ ]:
print("="*60)
print("AVERAGE MESSAGE LENGTH")
print("="*60)

for person in sorted(person_length):

    average = person_length[person] / person_messages[person]

    print(f"{person:10} : {average:.2f} characters")

AVERAGE MESSAGE LENGTH
Aman       : 26.72 characters
Karan      : 311.41 characters
Neha       : 25.37 characters
Priya      : 28.21 characters
Rahul      : 10.69 characters
Vikas      : 7.23 characters


In [ ]:
media_count = {}

for msg in messages:

    if msg["message"] == "<Media omitted>":

        sender = msg["sender"]

        if sender not in media_count:
            media_count[sender] = 0

        media_count[sender] += 1

In [ ]:
print("="*60)
print("MEDIA SHARED")
print("="*60)

for person in sorted(participants):

    count = media_count.get(person, 0)

    print(f"{person:10} : {count}")

MEDIA SHARED
Aman       : 4
Karan      : 7
Neha       : 8
Priya      : 4
Rahul      : 7
Vikas      : 2


In [ ]:


for msg in messages:

    msg["datetime"] = datetime.strptime(
        msg["timestamp"],
        "%d/%m/%y, %H:%M"
    )

In [ ]:
response_times = {}

for i in range(1, len(messages)):

    previous = messages[i-1]
    current = messages[i]

    if previous["sender"] == current["sender"]:
        continue

    gap = (current["datetime"] - previous["datetime"]).total_seconds()


    if gap > 21600:
        continue

    sender = current["sender"]

    if sender not in response_times:
        response_times[sender] = []

    response_times[sender].append(gap)

In [ ]:
print("="*60)
print("AVERAGE RESPONSE TIME")
print("="*60)

for person in sorted(participants):

    if person in response_times:

        avg = sum(response_times[person]) / len(response_times[person])

        minutes = avg / 60

        print(f"{person:10} : {minutes:.2f} minutes")

    else:

        print(f"{person:10} : No Replies")

AVERAGE RESPONSE TIME
Aman       : 55.36 minutes
Karan      : 36.62 minutes
Neha       : 39.45 minutes
Priya      : 41.99 minutes
Rahul      : 34.95 minutes
Vikas      : 46.30 minutes


In [ ]:
from datetime import timedelta

activity = {}

for person in participants:
    activity[person] = set()

for msg in messages:

    activity[msg["sender"]].add(
        msg["datetime"].date()
    )

In [ ]:
start = messages[0]["datetime"].date()
end = messages[-1]["datetime"].date()

print("="*60)
print("LONGEST SILENT STREAK")
print("="*60)

for person in sorted(participants):

    longest = 0
    current = 0

    day = start

    while day <= end:

        if day in activity[person]:

            current = 0

        else:

            current += 1

            if current > longest:
                longest = current

        day += timedelta(days=1)

    print(f"{person:10} : {longest} days")

LONGEST SILENT STREAK
Aman       : 0 days
Karan      : 0 days
Neha       : 0 days
Priya      : 0 days
Rahul      : 0 days
Vikas      : 11 days


In [ ]:
night_messages = {}

total_messages = {}

for person in participants:

    night_messages[person] = 0
    total_messages[person] = 0

for msg in messages:

    sender = msg["sender"]

    hour = msg["datetime"].hour

    total_messages[sender] += 1

    if hour >= 23 or hour <= 4:

        night_messages[sender] += 1

In [ ]:
print("="*60)
print("PERSONALITY ARCHETYPES")
print("="*60)

highest = max(person_count.values())

lowest = min(person_count.values())

for person in sorted(participants):

    total = total_messages[person]

    night = night_messages[person]

    night_percent = (night / total) * 100

    if person_count[person] == highest:

        title = "The Spammer"

    elif person_count[person] == lowest:

        title = "The Ghost"

    elif night_percent >= 75:

        title = "Night Owl"

    elif person == "Priya":

        title = "The Caretaker"

    elif person == "Karan":

        title = "The Storyteller"

    elif person == "Neha":

        title = "Drama Queen"

    else:

        title = "The Active Member"

    print(f"{person:10} --> {title}")

PERSONALITY ARCHETYPES
Aman       --> Night Owl
Karan      --> The Storyteller
Neha       --> Drama Queen
Priya      --> The Caretaker
Rahul      --> The Spammer
Vikas      --> The Ghost


In [ ]:
print("\n")
print("="*70)
print("               GROUPDNA REPORT")
print("="*70)

print("Group Name       : Hostel Bois 4ever")
print("Participants     :", len(participants))
print("Messages         :", len(messages))
print("Period           : 01 April 2024 - 30 May 2024")

print("\nTop Contributor")

name = max(person_count, key=person_count.get)

print(name, "-", person_count[name], "messages")

print("\nMost Silent")

name = min(person_count, key=person_count.get)

print(name, "-", person_count[name], "messages")

print("\nBusiest Day")

print(busiest_day)

print("\nBusiest Hour")

print(f"{busiest_hour:02}:00")

print("\nTop Word")

print(top_words[0][0])

print("\nProject Completed Successfully")



               GROUPDNA REPORT
Group Name       : Hostel Bois 4ever
Participants     : 6
Messages         : 3174
Period           : 01 April 2024 - 30 May 2024

Top Contributor
Rahul - 953 messages

Most Silent
Vikas - 24 messages

Busiest Day
04/05/24

Busiest Hour
18:00

Top Word
how

Project Completed Successfully
